# Categorical Value Normalization

## File information

- **Input:** `data/interim/cleaned_products_pricing_data.csv`
- **Output:** `data/processed/electronics_pricing_categorical_normalized.csv`
- **Purpose:** Standardize categorical text and create consistent values for analysis.
- **Original data:** Source categorical columns will be preserved; normalized values will be stored in new columns.

## Task checklist

- [ ] Load the cleaned interim dataset
- [ ] Normalize text-based categorical values
- [ ] Map condition and availability values to consistent groups
- [ ] Create a normalized category column
- [ ] Document the new columns and normalization rules
- [ ] Validate and save the normalized dataset


In [2]:
import pandas as pd

path = "../data/interim/cleaned_products_pricing_data.csv"
df = pd.read_csv(path)

print("Loaded file:", path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Loaded file: ../data/interim/cleaned_products_pricing_data.csv
Shape: (7249, 24)
Columns: ['id', 'prices.amountMax', 'prices.amountMin', 'prices.availability', 'prices.condition', 'prices.currency', 'prices.dateSeen', 'prices.isSale', 'prices.merchant', 'prices.sourceURLs', 'asins', 'brand', 'categories', 'dateAdded', 'dateUpdated', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'name', 'primaryCategories', 'sourceURLs', 'upc', 'weight']


## Categorical Column Inspection

Review the categorical columns before normalization. Check unique values, missing values, spelling, capitalization, whitespace, and inconsistent labels so the normalization rules are based on the actual data.


### 1. Inspect `prices.condition`

Check the condition values and their frequencies before cleaning. This identifies differences in capitalization, whitespace, missing values, and wording such as `New`, `new`, `Used`, or `Refurbished`.


In [3]:
if "prices.condition" not in df.columns:
    raise KeyError("Column 'prices.condition' was not found.")

print("Missing values:", df["prices.condition"].isna().sum())
print("Condition values:")
print(df["prices.condition"].value_counts(dropna=False))


Missing values: 0
Condition values:
prices.condition
New                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## 2. Clean `prices.condition`

The next code cell reads values from `prices.condition` and creates one new column: `condition_clean`.

It does all cleaning in one step:

- Removes extra spaces
- Treats `New`, `new`, and ` NEW ` as `new`
- Groups used and refurbished values
- Labels missing values as `unknown`
- Keeps the original `prices.condition` column unchanged


In [4]:
def clean_condition(value):
    if pd.isna(value):
        return "unknown"

    value = str(value).strip().casefold()
    if "refurbished" in value:
        return "refurbished"

    if value in {"pre-owned", "used"}:
        return "used"

    if value.startswith("new"):
        return "new"

    return "other"


df["condition_clean"] = df["prices.condition"].apply(clean_condition)
print(df["condition_clean"].value_counts(dropna=False))


condition_clean
new            7019
used            146
refurbished      83
other             1
Name: count, dtype: int64


### 3. Inspect `prices.availability`

Check the availability values and their frequencies before cleaning. This identifies differences in capitalization, whitespace, missing values, and wording such as `In Stock`, `Yes`, `Out of Stock`, or `Special Order`.


In [5]:
if "prices.availability" not in df.columns:
    raise KeyError("Column 'prices.availability' was not found.")

print("Missing values:", df["prices.availability"].isna().sum())
print("Availability values:")
print(df["prices.availability"].value_counts(dropna=False))


Missing values: 0
Availability values:
prices.availability
In Stock           3172
Yes                2136
yes                 893
TRUE                663
Out Of Stock        115
Special Order       109
More on the Way      91
undefined            40
sold                 22
No                    4
FALSE                 1
Retired               1
32 available          1
7 available           1
Name: count, dtype: int64


## 4. Clean `prices.availability`

Create one analytical output column, `availability_clean`, by trimming whitespace, standardizing case, and grouping values into `available`, `unavailable`, `limited/pending`, or `unknown`. The original column is preserved.


In [6]:
def clean_availability(value):
    if pd.isna(value):
        return "unknown"

    value = str(value).strip().casefold()
    if value in {"in stock", "yes", "true"} or value.endswith(" available"):
        return "available"

    if value in {"out of stock", "no", "false", "sold", "retired"}:
        return "unavailable"

    if value in {"special order", "more on the way"}:
        return "limited/pending"

    return "unknown"


df["availability_clean"] = df["prices.availability"].apply(clean_availability)
print(df["availability_clean"].value_counts(dropna=False))


availability_clean
available          6866
limited/pending     200
unavailable         143
unknown              40
Name: count, dtype: int64


### 5. Inspect `brand`

Check brand values and frequencies before cleaning. Look for missing values, extra spaces, and differences in capitalization or spelling.


In [7]:
if "brand" not in df.columns:
    raise KeyError("Column 'brand' was not found.")

print("Missing values:", df["brand"].isna().sum())
print("Brand values:")
print(df["brand"].value_counts(dropna=False).head(30))


Missing values: 0
Brand values:
brand
Sony               785
Samsung            744
Apple              248
Yamaha             240
Pioneer            176
LG                 175
Logitech           116
Lenovo             110
WD                 108
CORSAIR            103
Sennheiser          99
Alpine              93
Elite Screens       93
Corsair             92
Kenwood             91
SanDisk             91
Razer               86
Seagate             86
ASUS                81
Netgear             71
Panasonic           70
Onkyo               68
Denon               64
Lowepro             62
Alienware           62
JBL                 61
Dell                58
Canon               56
TP-Link             55
Western Digital     51
Name: count, dtype: int64


## 6. Clean `brand`

Create one output column, `brand_clean`, by removing extra spaces and standardizing case. The original `brand` column is preserved.


In [8]:
df["brand_clean"] = (
    df["brand"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .replace("", pd.NA)
)

print(df["brand_clean"].value_counts(dropna=False).head(30))


brand_clean
sony               785
samsung            744
apple              248
yamaha             240
corsair            195
pioneer            176
lg                 175
sandisk            138
logitech           116
asus               113
lenovo             110
wd                 108
sennheiser          99
alpine              93
elite screens       93
kenwood             91
jbl                 90
razer               86
seagate             86
netgear             82
panasonic           70
onkyo               68
v-moda              65
denon               64
lowepro             62
alienware           62
dell                58
tp-link             58
canon               56
house of marley     51
Name: count, dtype: Int64


### 7. Inspect `prices.merchant`

Check merchant values and frequencies before cleaning. Look for missing values, extra spaces, and differences in capitalization or spelling.


In [9]:
if "prices.merchant" not in df.columns:
    raise KeyError("Column 'prices.merchant' was not found.")

print("Missing values:", df["prices.merchant"].isna().sum())
print("Merchant values:")
print(df["prices.merchant"].value_counts(dropna=False).head(30))


Missing values: 0
Merchant values:
prices.merchant
Bestbuy.com                             2806
bhphotovideo.com                        1509
Walmart.com                              664
Beach Camera                             201
AMI Ventures Inc                          63
buydig                                    53
Focus Camera                              48
gear4less                                 43
echo-and-optics                           40
Beach Audio Inc                           39
Best Buy                                  38
UnbeatableSale                            33
DealClock                                 32
World Wide Stereo                         28
electronic_express                        28
BuyVPC                                    27
Electronics Expo (Authorized Dealer)      26
Car Audio Closeout                        24
Video & Audio Center                      19
OneCall                                   18
Electronic Express                        17
wwst

## 8. Clean `prices.merchant`

Create one generic merchant key, `merchant_clean`, by removing URL protocols, `www`, domain suffixes, URL paths, spaces, underscores, hyphens, punctuation, and case differences. This makes formatting variants of any merchant comparable. The original `prices.merchant` column is preserved.


In [14]:
import re


def clean_merchant(value):
    if pd.isna(value):
        return "unknown"

    value = str(value).strip().casefold()
    value = re.sub(r"^https?://", "", value)
    value = re.sub(r"^www\.", "", value)
    value = re.split(r"[/?#]", value, maxsplit=1)[0]
    value = re.sub(r"\.(com|net|org|co\.uk)$", "", value)
    value = re.sub(r"[^a-z0-9]", "", value)
    return value or "unknown"


df["merchant_clean"] = df["prices.merchant"].apply(clean_merchant)
print(df["merchant_clean"].value_counts(dropna=False).head(30))


merchant_clean
bestbuy                            2853
bhphotovideo                       1509
walmart                             664
beachcamera                         201
amiventuresinc                       63
focuscamera                          55
buydig                               53
electronicexpress                    45
gear4less                            43
echoandoptics                        42
beachaudioinc                        39
unbeatablesale                       33
dealclock                            32
worldwidestereo                      28
buyvpc                               27
electronicsexpoauthorizeddealer      26
onecall                              25
caraudiocloseout                     24
wholesaleconnection                  19
outletpc                             19
videoaudiocenter                     19
thepixelhub                          19
antonline                            18
wwstereo                             16
dell                     

### 9. Inspect `prices.shipping`

Check shipping values and frequencies before cleaning. Look for missing values and wording that indicates free, conditional free, or paid shipping.


In [ ]:
if "prices.shipping" not in df.columns:
    raise KeyError("Column 'prices.shipping' was not found.")

print("Missing values:", df["prices.shipping"].isna().sum())
print("Shipping values:")
print(df["prices.shipping"].value_counts(dropna=False).head(30))


## 10. Clean `prices.shipping`

Create one output column, `shipping_clean`, by grouping shipping descriptions into `free`, `conditional free`, `paid/other`, or `unknown`. The original `prices.shipping` column is preserved.


In [ ]:
def clean_shipping(value):
    if pd.isna(value):
        return "unknown"

    value = str(value).strip().casefold()
    if "free" in value:
        if "over" in value or "order" in value:
            return "conditional free"
        return "free"

    return "paid/other"


df["shipping_clean"] = df["prices.shipping"].apply(clean_shipping)
print(df["shipping_clean"].value_counts(dropna=False))


### 11. Inspect `prices.currency`

Check the currency values before deciding whether conversion is needed. Currency should not be converted without a documented exchange-rate rule.


In [15]:
if "prices.currency" not in df.columns:
    raise KeyError("Column 'prices.currency' was not found.")

currency_values = (
    df["prices.currency"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Currency values:")
print(currency_values.value_counts(dropna=False))

non_missing_currencies = set(currency_values.dropna())
if non_missing_currencies == {"USD"}:
    print("Decision: no currency transformation is needed; all values are USD.")
else:
    print("Decision: review conversion or filtering because multiple currencies are present.")


Currency values:
prices.currency
USD    7248
CAD       1
Name: count, dtype: Int64
Decision: review conversion or filtering because multiple currencies are present.


## 12. Keep USD listings

The dataset contains one CAD listing and 7,248 USD listings. Remove the CAD row rather than apply an undocumented exchange rate.


In [16]:
df = df[
    df["prices.currency"].astype("string").str.strip().str.upper() == "USD"
].copy()

print("Rows after removing non-USD listings:", len(df))
print("Currencies remaining:")
print(df["prices.currency"].value_counts(dropna=False))


Rows after removing non-USD listings: 7248
Currencies remaining:
prices.currency
USD    7248
Name: count, dtype: int64


## 13. Remove timestamps from date columns

Check every column containing `date` in its name and create a date-only column. The original timestamp columns are preserved for traceability.


In [ ]:
generated_duplicate_columns = [
    column for column in df.columns
    if column.endswith("_date_date")
]
if generated_duplicate_columns:
    df = df.drop(columns=generated_duplicate_columns)

date_columns = [
    column for column in df.columns
    if "date" in column.casefold() and not column.endswith("_date")
]

for column in date_columns:
    output_column = f"{column}_date"

    if column == "prices.dateSeen":
        def date_only_list(value):
            if pd.isna(value):
                return pd.NA
            dates = [
                pd.to_datetime(
                    item.strip(), errors="coerce", utc=True, format="mixed"
                ).date().isoformat()
                for item in str(value).split(",")
            ]
            return ", ".join(date for date in dates if date != "NaT") or pd.NA

        df[output_column] = df[column].apply(date_only_list)
    else:
        df[output_column] = pd.to_datetime(
            df[column], errors="coerce", utc=True, format="mixed"
        ).dt.date

print("Date columns found:", date_columns)
print("Date-only columns created:", [f"{column}_date" for column in date_columns])

Date columns found: ['prices.dateSeen', 'dateAdded', 'dateUpdated']
Date-only columns created: ['prices.dateSeen_date', 'dateAdded_date', 'dateUpdated_date']


0       2018-05-12
1       2018-06-13
2       2018-06-13
3       2018-06-13
4       2018-06-13
           ...    
7244    2018-06-13
7245    2018-06-13
7246    2018-06-13
7247    2018-06-13
7248    2018-06-13
Name: dateUpdated_date, Length: 7248, dtype: object

## 14. Inspect price columns

The dataset stores price ranges in `prices.amountMin` and `prices.amountMax`. Inspect both columns before deciding whether to create one representative `price` column.


In [24]:
required_price_columns = ["prices.amountMin", "prices.amountMax"]
missing_price_columns = [
    column for column in required_price_columns
    if column not in df.columns
]
if missing_price_columns:
    raise KeyError(f"Missing price columns: {missing_price_columns}")

df[required_price_columns] = df[required_price_columns].apply(
    pd.to_numeric, errors="coerce"
)

print("Missing values:")
print(df[required_price_columns].isna().sum())
print("Price summary:")
print(df[required_price_columns].describe())
print("Rows where minimum exceeds maximum:", (
    df["prices.amountMin"] > df["prices.amountMax"]
).sum())


Missing values:
prices.amountMin    0
prices.amountMax    0
dtype: int64
Price summary:
       prices.amountMin  prices.amountMax
count        7248.00000       7248.000000
mean          464.02150        495.593460
std           680.53323        763.633317
min             1.00000          1.000000
25%            79.95000         79.990000
50%           189.99000        198.990000
75%           479.99000        494.990000
max          5999.99000       6999.990000
Rows where minimum exceeds maximum: 0


## 15. Create one usable price column

Use the midpoint of `prices.amountMin` and `prices.amountMax` as the representative `price`. Inspect prices below `$5` and remove them because the project treats them as implausible electronics prices. The original price range columns are preserved.


In [25]:
df["price"] = (
    df["prices.amountMin"] + df["prices.amountMax"]
) / 2

print("Prices below $5:", (df["price"] < 5).sum())
print(df.loc[df["price"] < 5, [
    "name", "price", "prices.amountMin", "prices.amountMax"
]].sort_values("price").head(20))

# Remove implausibly low electronics prices according to the project rule.
df = df[df["price"] >= 5].copy()

print("Rows after price cleaning:", len(df))
print("Minimum price:", df["price"].min())
print("Maximum price:", df["price"].max())


Prices below $5: 13
                                                   name  price  \
483   Details About New Nib Novatel 6620l Verizon 4g...   1.00   
1111  Samsung 55 Class 4K (2160P) Smart LED TV (UN55...   1.00   
1711  Details About Alienware 13 R3 Aw13r3/13.3 Fhd/...   1.00   
2330  Details About Alienware 15 R3 Aw15r3/15.6 Fhd/...   1.00   
2444  Details About Razer Blade Laptop 14 Full Hd (i...   1.00   
2562  Dell - XPS 2-in-1 13.3 Touch-Screen Laptop - I...   1.00   
5464  Apple MacBook - 12 - Core m5 - 8 GB RAM - 512 ...   1.00   
3497  P-Series 55-Class UHD SmartCast LED Home Theat...   1.00   
5928  Apple MacBook Pro with Touch Bar - 13.3 - Core...   1.00   
3967  Optoma - UHD60 4K DLP Projector with High Dyna...   1.50   
4170  15.4 MacBook Pro with Touch Bar (Late 2016, Sp...   2.00   
3265  15.4 MacBook Pro with Touch Bar (Late 2016, Sp...   2.00   
3021  Apple iPhone SE Gold 16GB for Sprint ( MLY92LL...   2.99   

      prices.amountMin  prices.amountMax  
483         